# 事前準備：データのダウンロード
以下を実行して、外部ファイルをダウンロードしてください。   
**このセルはColaboratoryを起動するたびに必要となります**  

In [1]:
import urllib.request
import os
import sys

os.makedirs('text', exist_ok=True)
urllib.request.urlretrieve('https://park.itc.u-tokyo.ac.jp/yamakata-lab/lecture/mediaproc/mediaproc3/serohikino_goshu_org.txt', './text/serohikino_goshu_org.txt')

!pip install transformers
!pip install fugashi
!pip install unidic-lite
!pip install sentencepiece
!pip install sacremoses

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

# 第4回課題：BERTを使って「セロ弾きのゴーシュ」の最初の10文で一番ポジティブな文を見つけよう

## 入力データの説明

`text/serohikino_goshu_org.txt`には、宮沢賢治の「セロ弾きのゴーシュ」の全文が記録されています。   
全文をリスト`texts`に読み込み、最初の10文を書き出してみましょう。

In [2]:
infile = 'text/serohikino_goshu_org.txt'
texts = []
with open(infile, 'r', encoding = 'utf-8') as fi:
    for line in fi:
        texts.extend(line.rstrip('\n').replace('。', '。<EOS>').rstrip('<EOS>').split('<EOS>'))

texts[:10]

['ゴーシュは町の活動写真館でセロを弾く係りでした。',
 'けれどもあんまり上手でないという評判でした。',
 '上手でないどころではなく実は仲間の楽手のなかではいちばん下手でしたから、いつでも楽長にいじめられるのでした。',
 'ひるすぎみんなは楽屋に円くならんで今度の町の音楽会へ出す第六｜交響曲の練習をしていました。',
 'トランペットは一生けん命歌っています。',
 'ヴァイオリンも二いろ風のように鳴っています。',
 'クラリネットもボーボーとそれに手伝っています。',
 'ゴーシュも口をりんと結んで眼を皿のようにして楽譜を見つめながらもう一心に弾いています。',
 'にわかにぱたっと楽長が両手を鳴らしました。',
 'みんなぴたりと曲をやめてしんとしました。']

## 課題１：「セロ弾きのゴーシュ」の英文翻訳

リスト`texts`に保存されている「セロ弾きのゴーシュ」の最初の10文を、huggingfaceで公開されている機械翻訳モデル（`TopicAnalysis4.ipynb`で使用したモデル["Helsinki-NLP/opus-mt-ja-en"](https://huggingface.co/Helsinki-NLP/opus-mt-ja-en)）を使って日本語を英語に翻訳した結果を、リスト`translated`に保存してください。   
このとき、リスト`texts`の`n`番目の要素の翻訳結果がリスト`translated`の`n`番目の要素になるようにしてください。

In [3]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

# mp_ex4_1: ここにコードを書いてください。この行は消さないでください。
tokenizer_ja_en = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-ja-en")
model_ja_en = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-ja-en")

translated = []
for sentence in texts[:10]:
    input_ids = tokenizer_ja_en.encode(sentence, return_tensors="pt")
    translated_ids = model_ja_en.generate(input_ids)
    translated.append(tokenizer_ja_en.decode(translated_ids[0], skip_special_tokens=True))


/home/sou/git/media-programming/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 258/258 [00:00<00:00, 31658.83it/s]

変数 `translated` の最初の10文を書き出すと、以下のようになるはずです。  
```
["Gorsh was the guy who played celery in the city's active photo theater.",
 'But I had a reputation for not being very good at it.',
 "I wasn't very good at it, but in fact, I was the worst of all the other players, so I was always treated easily.",
 "Everyone who was too upset was circled into the dressing room and was practicing the first symphony for this town's concert.",
 'The trumpet is singing for life.',
 'Violet is also singing like a double wind.',
 'The clarinet also helps with the bowl.',
 'Gorsh, too, ties his mouth tightly together and looks like a plate in his eyes, and plays the music one more time as he looks at it.',
 'Suddenly, Kenji rang his hands.',
 'Everyone dropped the music quickly and tried to stop it.']
```
以下のセルで確認してください。

In [4]:
translated

["Gorsh was the guy who played celery in the city's active photo theater.",
 'But I had a reputation for not being very good at it.',
 "I wasn't very good at it, but in fact, I was the worst of all the other players, so I was always treated easily.",
 "Everyone who was too upset was circled into the dressing room and was practicing the first symphony for this town's concert.",
 'The trumpet is singing for life.',
 'Violet is also singing like a double wind.',
 'The clarinet also helps with the bowl.',
 'Gorsh, too, ties his mouth tightly together and looks like a plate in his eyes, and plays the music one more time.',
 'Suddenly, Kenji rang his hands.',
 'Everyone dropped the music quickly and tried to stop it.']

## 課題２：最もポジティブな文のインデックスの抽出

課題１で翻訳した英文10文に対し、一文ごとにTopicAnalysis4.ipynbで紹介したセンチメント分析のモデル"sentiment-analysis"を用いてセンチメントの傾向を予測し、最も `POSITIVE` と予測された英文のインデックスを変数  `maxid` に代入してください。   
モデルは[distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english)を使用してください。   
**評価では、変数`maxid`に正しい数値が代入されているかで正解しているかを判定します。**

In [5]:
classifier = pipeline("sentiment-analysis", model = 'distilbert-base-uncased-finetuned-sst-2-english')

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2667.48it/s]

In [6]:
# mp_ex4_2: ここにコードを書いてください。この行は消さないでください。
maxid = 0
max_score = 0.0
for i, sentence in enumerate(translated):
    result = classifier(sentence)[0]
    if result['label'] == 'POSITIVE' and result['score'] > max_score:
        max_score = result['score']
        maxid = i

print('maxid:', maxid)
print('positive score:', max_score)


maxid: 4
positive score: 0.9990360736846924


## 課題３：対応する英文が最もポジティブと判断と判断された和文の出力

課題2で最もポジティブと判断された英文に対応する和文を、 変数`rslt`に代入してからprintで書き出してください。   
**評価では、変数`rslt`に正しい和文が代入されているかで正解しているかを判定します。**

In [7]:
# mp_ex4_3: ここにコードを書いてください。この行は消さないでください。
rslt = texts[maxid]
print(rslt)


トランペットは一生けん命歌っています。


<!--2026S2-->